# Метод 2: символьные n-граммы и p-value

Готовая реализация находится в `m2.py`. Прогнозы оцениваются по корпусным меткам, не по исторически доказанному авторству

In [ ]:
from pathlib import Path
import json
import pandas as pd
root = Path('.')
metrics = json.loads((root/'results/metrics.json').read_text())
metadata = json.loads((root/'results/metadata.json').read_text())
pd.Series({k: metadata[k] for k in ['input_poems','retained_poems','n_training_fragments','n_features','temperature']})

## Независимые части

Обучение 317, калибровка вероятностей 106, conformal-калибровка 141, тест 142; Обучающие фрагменты нарезаются только после разбиения целых поэм. Словарь и IDF не видят проверку.

In [ ]:
pd.read_csv(root/'results/author_split_counts.csv')

In [ ]:
pd.Series({k:metrics[k] for k in ['n_poems','correct','accuracy','balanced_accuracy','macro_f1']})

## p-value метода

Перестановки меток целых произведений, затем переобучение; обучающие фрагменты одной поэмы имеют одну переставленную метку; тест условен на фиксированном очищенном корпусе и настройках; не учитывает прежний адаптивный выбор метода.

In [ ]:
pd.Series(metrics['permutation_test'])

## p-value произведения

Сравниваем `1 - probability` текста с независимыми conformal-эталонами того же автора. p = (1 + число эталонов с не меньшей нетипичностью)/(n_author + 1). 

In [ ]:
pred = pd.read_csv(root/'results/test_predictions.csv')
cols = ['key','author','predicted_author','probability_claimed','p_claimed','p_claimed_holm','set_size_005']
pred[cols].sort_values('p_claimed').head(12)

In [ ]:
counts = metrics['conformal']['counts_by_author']
pd.DataFrame([{'author':a,'n_calibration':v,'minimum_p':1/(v+1),'can_reject_at_005':1/(v+1)<=.05} for a,v in counts.items()])

## Запуск на реальном тексте

Пример - поэма Антары 187 из тестовой части. Она не участвует ни в обучении, ни в калибровке. Малое нескорректированное p — сигнал для дальнейшего рассмотрения

In [ ]:
from m2 import predict
predict(root/'results/m2_model.joblib', root/'example_query.json', root/'notebook_prediction.json')